In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import seaborn as sns
from datetime import datetime, timedelta
import random
import glob
import os
cwd = os.getcwd()
print(cwd)



In [ ]:
startTrad = 9.5 * 60 *60 
endTrad = 16.0 * 60 * 60
def load_data(ticker = 'AMZN', levels = '10', time_format = True, startTrad = startTrad, endTrad = endTrad):
    theMessageBookFileName = ticker + '_2025-10-06_34200000_57600000_message_' + levels + '.csv'

    theMessageBookFileName = os.path.join(cwd,'Lab 1', theMessageBookFileName)

    theMessageBook = pd.read_csv(theMessageBookFileName, 
                                    names = ['Time', 'Type', 'OrderID', 'Size', 'Price', 'TradeDirection', 'Unused'])

    theMessageBook['row_index'] = theMessageBook.index.values

    theMessageBookFiltered = theMessageBook[theMessageBook['Time'] >= startTrad]
    theMessageBookFiltered = theMessageBookFiltered[theMessageBookFiltered['Time'] <= endTrad]


    theOrderBookFileName = ticker + '_2025-10-06_34200000_57600000_orderbook_' + levels + '.csv'

    theOrderBookFileName = os.path.join(cwd,'Lab 1', theOrderBookFileName)

    col = ['Ask Price' , 'Ask Size', 'Bid Price', 'Bid Size']

    ColumnNames = []
    for level_no in range(1, int(levels) + 1):
        for col_name in col:
            ColumnNames.append(col_name + ' ' + str(level_no))

    theOrderBook = pd.read_csv(theOrderBookFileName, 
                                    names = ColumnNames)
    
    theOrderBookFiltered = theOrderBook.iloc[theMessageBookFiltered.row_index]

    if time_format:  #function argument
        theMessageBookFiltered = theMessageBookFiltered.set_index(pd.to_datetime(theMessageBookFiltered.Time, unit='s'))

    theOrderBookFiltered = theOrderBookFiltered.set_index(theMessageBookFiltered.index)

    data_col_dic = {}

    for c in col:
        key = c.strip()
        value = []
        for level_no in range(1, int(levels) + 1):
            value.append(c + ' ' + str(level_no))
        data_col_dic[key] = value
    
    Orderbook_dic = {}

    for key in data_col_dic:
        cols_for_key = data_col_dic[key]
        Orderbook_dic[key] = theOrderBookFiltered[cols_for_key]

    return theMessageBookFiltered, Orderbook_dic 

In [ ]:
msg_book_AAPL, OB_AAPL = load_data(ticker = 'AAPL', levels = '10')
msg_book_AAPL, OB_AAPL = load_data(ticker = 'AAPL', levels = '10')
msg_book_AMZN, OB_AMZN = load_data(ticker = 'AMZN', levels = '50')
msg_book_BAC, OB_BAC = load_data(ticker = 'BAC', levels = '10')
msg_book_GOOG, OB_GOOG = load_data(ticker = 'GOOG', levels = '10')
msg_book_TSLA, OB_TSLA = load_data(ticker = 'TSLA', levels = '10')                                 


In [ ]:
OB_AAPL['Ask Price'].head(1)

In [ ]:
# Function to see a random snapshot of the LOB

def order_book_look(OB_dic, ticker, t_star):
    # Note: Pick a random row/event from the order book if not specified

    idx = OB_dic['Bid Size'].index

    # find first row with time >= t_star
    i = 0
    while i < len(idx) and idx[i] < t_star:
        i += 1

    if i == len(idx):
        raise ValueError(f"{ticker}: no data at/after t_star")

    bid_data   = OB_dic['Bid Size'].iloc[i].to_numpy(dtype=float)
    bid_prices = OB_dic['Bid Price'].iloc[i].to_numpy(dtype=float)

    ask_data   = OB_dic['Ask Size'].iloc[i].to_numpy(dtype=float)
    ask_prices = OB_dic['Ask Price'].iloc[i].to_numpy(dtype=float)

    tstamp = idx[i]
    time = tstamp.strftime('%H:%M:%S:%f')

    ## Plot 1 - Snapshot of the Limit Order Book

    fig, ax = plt.subplots(2,1, figsize = (9,9))
    
    ax[0].bar(ask_prices/10000, ask_data,
              width = 0.007, color='#fc1b04', label='Ask')
    # Divide price by 10000 to get price in dollars
    ax[0].bar(bid_prices/10000,bid_data,
              width=0.007,color='#13fc04', label='Bid')
    ax[0].set_ylabel('Quantity')
    ax[0].set_xlabel('Price($)')
    ax[0].set_title(f'Order book at {str(time)} for {ticker}')


    l = len(bid_data)

    # Plot 2 - Relative Depth in the Limit Order Book
    #----------------------------------------------------------------------------
    # Plot variables 
    # adding up size of orders to get cumulative depth and normalise by total side volume
    ax[1].step(range(1,l + 1,1),bid_data.cumsum()/bid_data.sum(),label='Bid',color='#13fc04')  #step plot
    # -1 to get values below axis
    ax[1].step(range(1,l + 1,1),-1*ask_data.cumsum()/ask_data.sum(),label='Ask',color='#fc1b04')

    ax[1].set_ylim(-1,1)
    ax[1].set_xlim(1,10)
    ax[1].set_title('Relative Depth in the Limit Order Book for ' + ticker + ' at ' + str(time))
    ax[1].set_ylabel('% Orderbook')
    ax[1].set_xlabel('Level')

    return i

In [ ]:

idx_ref = OB_AAPL['Bid Size'].index
t_star = idx_ref[np.random.randint(len(idx_ref))]

orderbooks = {
    "AAPL": OB_AAPL,
    "AMZN": OB_AMZN,
    "BAC":  OB_BAC,
    "GOOG": OB_GOOG,
    "TSLA": OB_TSLA,
}

for ticker, OB in orderbooks.items():
    order_book_look(OB, ticker, t_star)

### Q2) Tick Size Classification Use the quantitative criteria introduced in class to classify each stock as a small, medium or large tick stock

That is for each asset:

Compute at least two different metrics such as average relative spread in ticks, Fraction of time the spread equals one tick, quoted depth at the best bed and ask.

Then use your numerical findings to justify the classification


In [ ]:
# For Analysis we would like to get the message book and order book in one dataframe

# First view both dataframes

msg_book_AAPL.head(1)

In [ ]:
OB_AAPL['Ask Size'].iloc[0:1]
# OB_AAPL['Ask Price'].head(1)

In [ ]:
print(f" The number of rows of the LOB is {len(OB_AAPL['Ask Price'])}")
print(f"The number of rows of the message book is {len(msg_book_AAPL)}")

In [ ]:
def interleave_prices_sizes(prices_df, sizes_df, side = 'Bid'):
    n_levels = prices_df.shape[1] # number of columns
    columns = []
    for i in range(n_levels):
        price_col = prices_df.iloc[:, i] # selects all rows in column i
        size_col = sizes_df.iloc[:, i]
        pair_df = pd.concat([price_col, size_col], axis = 1)
        pair_df.columns = [f"{side} Price {i+ 1}", f"{side} Size {i + 1}"] # assign column names
        columns.append(pair_df)
    return pd.concat(columns, axis = 1)

# AAPL
AAPL_bid = interleave_prices_sizes(OB_AAPL['Bid Price'], OB_AAPL['Bid Size'], side='Bid')
AAPL_ask = interleave_prices_sizes(OB_AAPL['Ask Price'], OB_AAPL['Ask Size'], side='Ask')
AAPL_LOB = pd.concat([AAPL_bid, AAPL_ask], axis=1)
merged_AAPL = pd.concat([msg_book_AAPL.reset_index(drop=True), AAPL_LOB.reset_index(drop=True)], axis=1)

# AMZN
AMZN_bid = interleave_prices_sizes(OB_AMZN['Bid Price'], OB_AMZN['Bid Size'], side='Bid')
AMZN_ask = interleave_prices_sizes(OB_AMZN['Ask Price'], OB_AMZN['Ask Size'], side='Ask')
AMZN_LOB = pd.concat([AMZN_bid, AMZN_ask], axis=1)
merged_AMZN = pd.concat([msg_book_AMZN.reset_index(drop=True), AMZN_LOB.reset_index(drop=True)], axis=1)

# BAC
BAC_bid = interleave_prices_sizes(OB_BAC['Bid Price'], OB_BAC['Bid Size'], side='Bid')
BAC_ask = interleave_prices_sizes(OB_BAC['Ask Price'], OB_BAC['Ask Size'], side='Ask')
BAC_LOB = pd.concat([BAC_bid, BAC_ask], axis=1)
merged_BAC = pd.concat([msg_book_BAC.reset_index(drop=True), BAC_LOB.reset_index(drop=True)], axis=1)

# GOOG
GOOG_bid = interleave_prices_sizes(OB_GOOG['Bid Price'], OB_GOOG['Bid Size'], side='Bid')
GOOG_ask = interleave_prices_sizes(OB_GOOG['Ask Price'], OB_GOOG['Ask Size'], side='Ask')
GOOG_LOB = pd.concat([GOOG_bid, GOOG_ask], axis=1)
merged_GOOG = pd.concat([msg_book_GOOG.reset_index(drop=True), GOOG_LOB.reset_index(drop=True)], axis=1)

# TSLA
TSLA_bid = interleave_prices_sizes(OB_TSLA['Bid Price'], OB_TSLA['Bid Size'], side='Bid')
TSLA_ask = interleave_prices_sizes(OB_TSLA['Ask Price'], OB_TSLA['Ask Size'], side='Ask')
TSLA_LOB = pd.concat([TSLA_bid, TSLA_ask], axis=1)
merged_TSLA = pd.concat([msg_book_TSLA.reset_index(drop=True), TSLA_LOB.reset_index(drop=True)], axis=1)


In [ ]:
merged_AAPL.head(1)

In [ ]:
# Average Relative Spread 
NSDQ_tick = 0.01 * 10000

orderbooks = {
    "AAPL": merged_AAPL,
    "AMZN": merged_AMZN,
    "BAC":  merged_BAC,
    "GOOG": merged_GOOG,
    "TSLA": merged_TSLA
}

def compute_average_relative_spread(OB):
    
    ask = OB['Ask Price 1']
    bid = OB['Bid Price 1']
    OB['Spread'] = (ask - bid)
    OB['Relative Spread'] = (ask - bid) / NSDQ_tick

    return OB['Relative Spread'].dropna().mean()

def classify_stock_by_relative_spread(avg_rel_spread_ind):

    if avg_rel_spread_ind <= 1.5:
        return "Large-tick"
    elif (avg_rel_spread_ind >1.5) and (avg_rel_spread_ind)<= 3 :
        return "Medium-tick"
    else:
        return "Small-tick"#
    

def classify_all_stocks(orderbooks):

    for ticker, OB in orderbooks.items():
        avg_spread = compute_average_relative_spread(OB)
        classification = classify_stock_by_relative_spread(avg_spread)

        print(
            f"{ticker}: "
            f"average relative spread = {avg_spread:.3f} ticks → {classification}"
        )

classify_all_stocks(orderbooks)

Second Classification metric Fraction of time the spread equals one tick

In [ ]:
merged_AAPL['Spread of one tick'] = merged_AAPL['Spread'].between(95, 105)
merged_BAC['Spread of one tick']  = merged_BAC['Spread'].between(95, 105)
merged_AMZN['Spread of one tick'] = merged_AMZN['Spread'].between(95, 105)
merged_GOOG['Spread of one tick'] = merged_GOOG['Spread'].between(95, 105)
merged_TSLA['Spread of one tick'] = merged_TSLA['Spread'].between(95, 105)


def fraction_of_time_spread_equals_one(ticker, df):
    time_spent_one = df['Spread of one tick'].sum()

    percentage_time = (time_spent_one/ (df.shape[0] - 1)) * 100

    print(f' {ticker} spent {percentage_time:.3f} % of its time with a spread approximately one tick')

for ticker, OB in orderbooks.items():
    fraction_of_time_spread_equals_one(ticker, OB)



The final metric the question asks for is the quoted depth at the best bid and ask, I will make these a proportion of the total depth at bid and ask side and average this over the day

In [ ]:
def best_depth_concentration(ticker, df, n_levels = 10):
    df['Depth at Best'] = df['Bid Size 1'] + df['Ask Size 1']
    bid_cols = [f'Bid Size {i}' for i in range(1, n_levels + 1)]
    ask_cols = [f'Ask Size {i}' for i in range(1, n_levels + 1)]

    df['Total Depth at Levels 1-10'] = df[bid_cols + ask_cols].sum(axis = 1)

    df['Proportion of Depth at Best']  = df['Depth at Best'] / df['Total Depth at Levels 1-10']

    average_depth_at_best = df['Proportion of Depth at Best'].mean() * 100

    print(f' {average_depth_at_best:.3f} % of total depth is at best on average')

for ticker, OB in orderbooks.items():
    best_depth_concentration(ticker, OB)

## Question 3)  Average Spread and First Gap Analysis

For each stock, define and comput the average of the following quantities over the entire trading day:
- The average bid-ask spread
- The average first gap on the ask side

a) create a scatter plot where each point corresponds to a stock
b) the x-axis should represent the average spread adn the y axis the average first gap
c) label each point by ticker

Comment on the relationship between spread and first gap. What does it tell us avbout liquidity and book structure?

In [ ]:
Average_Spread_first_gap = {}

def calculate_average_bid_ask_spread(ticker, df):

    average_spread = df['Spread'].mean()

    df['First Ask Gap'] = df['Ask Price 2'] - df['Ask Price 1']

    average_first_gap = df['First Ask Gap'].mean()

    Average_Spread_first_gap[ticker] = [average_spread, average_first_gap]


for ticker, OB in orderbooks.items():
    calculate_average_bid_ask_spread(ticker, OB)

Average_Spread_first_gap


tickers = list(Average_Spread_first_gap.keys())

avg_spread = np.array([Average_Spread_first_gap[t][0] for t in tickers])
avg_first_gap = np.array([Average_Spread_first_gap[t][1] for t in tickers])


In [ ]:
plt.figure(figsize= (7,5))
plt.scatter(avg_spread, avg_first_gap)

for i, ticker in enumerate(tickers):
   plt.annotate(ticker, (avg_spread[i], avg_first_gap[i]), textcoords = "offset points", xytext=(5,5))
   

plt.xlabel("Average bid-ask spread")
plt.ylabel("Average first ask gap")
plt.title("Average Spread vs First Ask Gap")
plt.grid(True)
plt.show()

Large Tick Assets exhibit both smaller average spreads and smaller first-level gaps, reflecting that most of the depth is at the best and price priority is prevelant in this market microstructure due to the large cost of price priority

Small Tick Assets exhibit both larger average spreads and larger first-level gaps.


### Q4) Depth at the Best Level
For each stock:
- Extract the depth at the best bid and best ask price levels (Level 1) over the full trading day:
-  $ Depth^{bid}_t = Bid Volume_1(t)$ ,             $Depth^{ask}_t = Ask Volume_1(t)$
-  Plot both time series over the day 
-  Compute and report the average depth at best one each side over the trading day. 

In [ ]:
def extract_depth_at_best(ticker, OB, rolling_window=300):
    Best_df = OB[['Time', 'Bid Size 1', 'Ask Size 1']].copy()

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    Best_df['Bid Size 1'].plot(ax=ax[0], alpha=0.4,
                               title=f'{ticker} Best Bid Size over time')
    Best_df['Ask Size 1'].plot(ax=ax[1], alpha=0.4,
                               title=f'{ticker} Best Ask Size over time')

    Best_df['Time'] = pd.to_datetime(Best_df['Time'], unit='s', origin='unix')

    avg_bid = Best_df.resample(f'{rolling_window}s', on='Time', label='right')['Bid Size 1'].mean()
    avg_ask = Best_df.resample(f'{rolling_window}s', on='Time', label='right')['Ask Size 1'].mean()

    Best_df['Average Bid Size'] = avg_bid.reindex(Best_df['Time'], method='ffill').values
    Best_df['Average Ask Size'] = avg_ask.reindex(Best_df['Time'], method='ffill').values

    Best_df['Average Bid Size'].plot(ax=ax[0], color='orange', linewidth=2, zorder=3)
    Best_df['Average Ask Size'].plot(ax=ax[1], color='orange', linewidth = 2, zorder=3)

    plt.tight_layout()
    return Best_df

for ticker, OB in orderbooks.items():
    extract_depth_at_best(ticker, OB, rolling_window=300)
